In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
import os,re,random,json,pickle,time,math,zipfile,shutil
from collections import Counter,defaultdict
from dataclasses import dataclass, field
from datasets import Dataset as HFDataset
import math,json,os,time,glob
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as checkpoint
from torch.utils.data import Dataset, DataLoader
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm
import multiprocessing as mp

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful")

In [ ]:
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

In [ ]:
hf_dataset_repo = "Exploration-Lab/CS-781-Capstone"
test_file = "Phase_1_Pretraining/Test/test.json"
train_file = "Phase_1_Pretraining/Train/train_supreme_court.json"


@dataclass
class data_configuration:
    train_tokens: int = 200_000_000
    val_tokens: int = 3_000_000
    val_fraction: float = 0.2
    tokenizer_train_docs_per_source: int = 20000
    seed: int = 29


@dataclass
class tokenizer_configuration:
    vocab_size: int = 32000
    tokenizer_path: str = "/kaggle/working/legal_bpe_tokenizer.pkl"
    num_workers: int = 4


@dataclass
class arch_configuration:
    name: str = "decoder_v7"
    context_length: int = 1536     
    n_embd: int = 768               
    n_layer: int = 12             
    n_head: int = 12                
    target_param_range: tuple = (50_000_000, 300_000_000)


@dataclass
class training_configuration:
    epochs: int = 3
    per_device_train_batch_size: int = 8
    per_device_eval_batch_size: int = 8
    gradient_accumulation_steps: int = 4
    learning_rate: float = 3e-4
    weight_decay: float = 0.1
    warmup_steps: int = 500
    lr_scheduler_type: str = "cosine"
    fp16: bool = True
    eval_steps: int = 500
    logging_steps: int = 50
    save_steps: int = 500
    save_total_limit: int = 3
    seed: int = 29


@dataclass
class generation_configuration:
    target_generation_words: int = 128
    max_generation_tokens: int = 384
    temperature: float = 0.7
    top_p: float = 0.95
    top_k: int = 50
    do_sample: bool = True
    

data_cfg = data_configuration()
tok_cfg = tokenizer_configuration()
arch_cfg = arch_configuration()
train_cfg = training_configuration()
gen_cfg = generation_configuration()


In [ ]:
train_file_path = hf_hub_download(repo_id=hf_dataset_repo,filename=train_file,repo_type="dataset",token=HF_TOKEN)

print("Training file:")
print(train_file_path)
with open(train_file_path,"r",encoding="utf-8") as f:
    raw_data = json.load(f)

print(len(raw_data))

In [ ]:
WORD_END=256
RESERVED_TOKENS=["<pad>","<unk>","<s>","</s>"]

def pre_tokenize_preserve(text):
    return re.findall(r"\w+|[^\w\s]|\s+", text)

def word_to_symbols(word):
    byte_values=word.encode("utf-8")
    symbols=[(b,) for b in byte_values]
    symbols.append((WORD_END,))
    return tuple(symbols)

def symbol_to_string(symbol):
    contain_end_token=WORD_END in symbol
    byte_nums=[b for b in symbol if b!=WORD_END]
    text=bytes(byte_nums).decode("utf-8",errors="replace")
    if contain_end_token:
        text+="</w>"
    return text

def train_bpe(corpus_texts,vocab_size=32000,verbose_every=None):
    word_counts=Counter()
    for text in tqdm(corpus_texts,desc="pre-tokenizing documents"):
        for unit in pre_tokenize_preserve(text):
            word_counts[unit]+=1

    word_freqs=Counter()
    for word,count in word_counts.items():
        symbols=word_to_symbols(word)
        word_freqs[symbols]+=count

    itos_base=list(RESERVED_TOKENS)
    for byte_value in range(256):
        itos_base.append((byte_value,))
    itos_base.append((WORD_END,))
    stoi={symbol:idx for idx,symbol in enumerate(itos_base)}
    base_size=len(itos_base)
    num_merges=vocab_size-base_size

    pair_counts = Counter()
    pair_to_words = defaultdict(set)
    for symbols,freq in tqdm(word_freqs.items(),total=len(word_freqs),desc="building pair index"):
        for i in range(len(symbols)-1):
            pair=(symbols[i],symbols[i+1])
            pair_counts[pair]+=freq
            pair_to_words[pair].add(symbols)

    merges=[]
    pbar=tqdm(total=num_merges,desc="training bpe merges")
    for x in range(num_merges):
        if not pair_counts:
            pbar.write(f"no pairs left, stopping at {x} merges")
            break
        highest_freq=max(pair_counts.values())
        if highest_freq<=1:
            pbar.write(f"best pair freq is {highest_freq},stopping at {x} merges")
            break

        tied_pairs=[p for p,freq in pair_counts.items() if freq==highest_freq]
        chosen_pair=min(tied_pairs)
        sa,sb=chosen_pair
        merged_symbol=sa+sb

        affected_words=list(pair_to_words.get(chosen_pair,()))
        for old_symbols in affected_words:
            freq=word_freqs.get(old_symbols)
            if freq is None:
                continue

            for i in range(len(old_symbols)-1):
                p=(old_symbols[i],old_symbols[i+1])
                pair_counts[p]-=freq
                if pair_counts[p]<=0:
                    del pair_counts[p]
                pair_to_words[p].discard(old_symbols)
                if not pair_to_words[p]:
                    del pair_to_words[p]

            new_symbols,i=[],0
            while i<len(old_symbols):
                pair_found=(i<len(old_symbols)-1 and old_symbols[i]==sa and old_symbols[i+1]==sb)
                if pair_found:
                    new_symbols.append(merged_symbol)
                    i+=2
                else:
                    new_symbols.append(old_symbols[i])
                    i+=1
            new_symbols=tuple(new_symbols)

            del word_freqs[old_symbols]
            word_freqs[new_symbols]+=freq

            for i in range(len(new_symbols) - 1):
                p = (new_symbols[i], new_symbols[i + 1])
                pair_counts[p] += freq
                pair_to_words[p].add(new_symbols)

        merges.append(chosen_pair)
        itos_base.append(merged_symbol)
        stoi[merged_symbol]=len(itos_base)-1
        pbar.update(1)

    pbar.close()

    merge_rank={pair:rank for rank,pair in enumerate(merges)}
    itos=[]
    for symbol in itos_base:
        if isinstance(symbol,str):
            itos.append(symbol)
        else:
            itos.append(symbol_to_string(symbol))

    bpe_data={"stoi": stoi,"itos": itos,"itos_raw": itos_base,"merges": merges,"merge_rank": merge_rank,"vocab_size": len(itos_base)}
    return bpe_data

In [ ]:
def get_adjacent_pairs(symbols):
    return set(zip(symbols, symbols[1:]))

def apply_bpe_merges(symbols,merge_rank):
    symbols=list(symbols)
    while len(symbols)>1:
        pairs=get_adjacent_pairs(symbols)
        candidates=[p for p in pairs if p in merge_rank]
        if not candidates:
            break
        best_pair=min(candidates,key=lambda p:merge_rank[p])
        sa,sb=best_pair
        merged=sa+sb
        new_symbols,i=[],0
        while i<len(symbols):
            if i<len(symbols)-1 and symbols[i]==sa and symbols[i+1]==sb:
                new_symbols.append(merged)
                i+=2
            else:
                new_symbols.append(symbols[i])
                i+=1
        symbols=new_symbols
    return symbols

def make_encoder(bpe_data):
    stoi=bpe_data["stoi"]
    merge_rank=bpe_data["merge_rank"]
    cache={}

    def encode(text,add_eos=False):
        ids=[]
        for unit in pre_tokenize_preserve(text):
            if unit not in cache:
                symbols=word_to_symbols(unit)
                final_symbols=apply_bpe_merges(symbols,merge_rank)
                cache[unit]=[stoi[s] for s in final_symbols]
            ids.extend(cache[unit])
        if add_eos:
            ids.append(stoi["</s>"])
        return ids
    return encode

def decode(ids,bpe_data,skip_special_tokens=True):
    itos_raw=bpe_data["itos_raw"]
    buf=bytearray()
    for i in ids:
        symbol=itos_raw[i]
        if isinstance(symbol, str):    
            if not skip_special_tokens:
                buf.extend(symbol.encode("utf-8"))
            continue
        for b in symbol:
            if b!=WORD_END:
                buf.append(b)
    return buf.decode("utf-8",errors="replace")

In [ ]:
def load_training_texts(raw_data, seed, max_docs=None):
    print(max_docs)
    rng = random.Random(seed)
    texts = []
    data_copy = raw_data[:]         
    rng.shuffle(data_copy)
    if max_docs is not None:
        data_copy = data_copy[:max_docs]
    for data in data_copy:
        text = data.get("text", "")
        if text:
            texts.append(text)
    return texts

if os.path.exists(tok_cfg.tokenizer_path):
    with open(tok_cfg.tokenizer_path, "rb") as f:
        bpe_data = pickle.load(f)
    print("Loaded existing tokenizer from", tok_cfg.tokenizer_path)
else:
    training_texts = load_training_texts(raw_data,data_cfg.seed,data_cfg.tokenizer_train_docs_per_source)
    print(f"Training BPE on {len(training_texts)} documents,target vocab_size={tok_cfg.vocab_size}")
    bpe_data = train_bpe(training_texts, vocab_size=tok_cfg.vocab_size)
    with open(tok_cfg.tokenizer_path, "wb") as f:
        pickle.dump(bpe_data, f)
    print("Saved to", tok_cfg.tokenizer_path)

VOCAB_SIZE = bpe_data["vocab_size"]
PAD_ID = bpe_data["stoi"]["<pad>"]
EOS_ID = bpe_data["stoi"]["</s>"]
print("Vocabulary size:", VOCAB_SIZE)

_sample = "Section 302(1) of the IPC, 1860 was invoked in AIR 1973 SC 1461."
_ids = make_encoder(bpe_data)(_sample)
_back = decode(_ids, bpe_data)
print("Round-trip OK:" , _sample == _back)
print("  original:", repr(_sample))
print("  decoded :", repr(_back))

In [ ]:
_worker_encode = None

def _init_worker(bpe_data):
    global _worker_encode
    _worker_encode = make_encoder(bpe_data)

def _encode_one(text):
    return _worker_encode(text, add_eos=True)

def encode_texts_parallel(texts, bpe_data, num_workers=4, chunksize=10):
    if not texts:
        return []
    with mp.Pool(processes=num_workers, initializer=_init_worker, initargs=(bpe_data,)) as pool:
        results = pool.map(_encode_one, texts, chunksize=chunksize)
    return results  

def split_train_val_records(raw_data, val_fraction, seed):

    rng = random.Random(seed)
    data_copy = raw_data[:]
    rng.shuffle(data_copy)
    n_val = max(1, int(len(data_copy) * val_fraction))
    val_records = data_copy[:n_val]
    train_records = data_copy[n_val:]
    return train_records, val_records


def tokenize_corpus(records, token_budget, bpe_data, seed, num_workers=4, batch_size=200):
    rng = random.Random(seed)
    recs = records[:]
    rng.shuffle(recs)

    collected = 0
    all_tokens = []
    pbar = tqdm(total=token_budget, unit="tok", unit_scale=True, desc="Tokenizing")
    for start in range(0, len(recs), batch_size):
        if collected >= token_budget:
            break
        batch = recs[start:start + batch_size]
        texts = [r.get("text", "") for r in batch]
        batch_ids = encode_texts_parallel(texts, bpe_data, num_workers=num_workers)
        for ids in batch_ids:
            remaining = token_budget - collected
            if remaining <= 0:
                break
            n_take = min(len(ids), remaining)
            all_tokens.extend(ids[:n_take])
            collected += n_take
            pbar.update(n_take)
    pbar.close()

    if collected < token_budget:
        print(f"  Ran out of records: requested {token_budget:,} tokens, got {collected:,}")
    return all_tokens


def create_blocks(tokens, block_size):
    total_length = (len(tokens) // block_size) * block_size
    tokens = tokens[:total_length]
    return [tokens[i:i + block_size] for i in range(0, total_length, block_size)]


train_records, val_records = split_train_val_records(raw_data, data_cfg.val_fraction, data_cfg.seed)
print(f"Train records: {len(train_records)} | Val records: {len(val_records)}")

print("Building train token stream...")
train_tokens = tokenize_corpus(train_records, data_cfg.train_tokens, bpe_data,
                                data_cfg.seed, num_workers=tok_cfg.num_workers)
print("Building val token stream...")
val_tokens = tokenize_corpus(val_records, data_cfg.val_tokens, bpe_data,data_cfg.seed + 1, num_workers=tok_cfg.num_workers)

print(f"Train tokens: {len(train_tokens):,} | Val tokens: {len(val_tokens):,}")

train_blocks = create_blocks(train_tokens, arch_cfg.context_length)
val_blocks = create_blocks(val_tokens, arch_cfg.context_length)
print(f"Train sequences: {len(train_blocks)} | Val sequences: {len(val_blocks)}")

In [ ]:
train_dataset = HFDataset.from_dict({"input_ids": train_blocks})
val_dataset = HFDataset.from_dict({"input_ids": val_blocks})

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm_x = x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return norm_x * self.weight

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len, base=10000.0):
        super().__init__()
        assert head_dim % 2 == 0, "RoPE requires an even head_dim"
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self.max_seq_len = max_seq_len
        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, dtype=self.inv_freq.dtype, device=self.inv_freq.device)
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def forward(self, seq_len, offset, device, dtype):
        needed = offset + seq_len
        if needed > self.cos_cached.shape[0]:
            self._build_cache(needed)
        cos = self.cos_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        sin = self.sin_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        return cos, sin

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rope(x, cos, sin):
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return (x * cos) + (rotate_half(x) * sin)

class GroupedQueryAttention(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, context_length, dropout=0.1):
        super().__init__()
        assert n_embd % n_head == 0
        assert n_head % n_kv_head == 0, 
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.n_rep = n_head // n_kv_head
        self.head_dim = n_embd // n_head

        self.q_proj = nn.Linear(n_embd, n_head * self.head_dim, bias=False)
        self.k_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.v_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)

        self.attn_dropout_p = dropout
        self.resid_dropout = nn.Dropout(dropout)
        self.context_length = context_length
        self.rope = RotaryEmbedding(self.head_dim, context_length)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False, position_offset=0):
        B, T, C = x.shape

        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)

        cos, sin = self.rope(T, position_offset, x.device, x.dtype)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        present = (k, v) if use_cache else None

      
        k_exp = k.repeat_interleave(self.n_rep, dim=1)
        v_exp = v.repeat_interleave(self.n_rep, dim=1)

        dropout_p = self.attn_dropout_p if self.training else 0.0
        if attn_mask is not None:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, attn_mask=attn_mask, dropout_p=dropout_p)
        elif past_kv is not None:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, dropout_p=dropout_p)
        else:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, dropout_p=dropout_p, is_causal=True)

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present


class MLP(nn.Module):
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class Block(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, context_length, dropout=0.1):
        super().__init__()
        self.ln_1 = RMSNorm(n_embd)
        self.attn = GroupedQueryAttention(n_embd, n_head, n_kv_head, context_length, dropout)
        self.ln_2 = RMSNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False, position_offset=0):
        attn_out, present = self.attn(self.ln_1(x), attn_mask, past_kv, use_cache, position_offset)
        x = x + attn_out
        x = x + self.mlp(self.ln_2(x))
        return (x, present) if use_cache else x

class DecoderOnlyLM_RMSNorm_GQA(nn.Module):
    def __init__(self, vocab_size, context_length, n_embd, n_layer, n_head, n_kv_head,
                dropout=0.1, pad_id=0, gradient_checkpointing=False):
        super().__init__()
        self.vocab_size = vocab_size
        self.context_length = context_length
        self.pad_id = pad_id
        self.gradient_checkpointing = gradient_checkpointing
        self.wte = nn.Embedding(vocab_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            Block(n_embd, n_head, n_kv_head, context_length, dropout) for _ in range(n_layer)
        ])
        self.ln_f = RMSNorm(n_embd)

        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, attention_mask=None, labels=None,
                past_key_values=None, use_cache=False, last_only=False):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        assert past_len + T <= self.context_length, \
            f"seq len {past_len + T} > context_length {self.context_length}"

        combined_mask = None
        if attention_mask is not None:
            key_mask = attention_mask.bool()[:, None, None, :]
            if past_len == 0:
                causal = torch.tril(torch.ones(T, T, dtype=torch.bool, device=input_ids.device))
                eye = torch.eye(T, dtype=torch.bool, device=input_ids.device)
                combined_mask = (causal[None, None] & key_mask) | eye[None, None]
            else:
                combined_mask = key_mask

        x = self.drop(self.wte(input_ids))

        presents = [] if use_cache else None
        for i, block in enumerate(self.blocks):
            past = past_key_values[i] if past_key_values is not None else None
            if self.gradient_checkpointing and self.training and not use_cache:
                x = checkpoint.checkpoint(block, x, combined_mask, None, False, past_len,
                                          use_reentrant=False)
            else:
                out = block(x, combined_mask, past, use_cache, past_len)
                if use_cache:
                    x, present = out
                    presents.append(present)
                else:
                    x = out

        x = self.ln_f(x)
        if last_only:
            x = x[:, -1:, :]
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=self.pad_id,
            )
        return {"loss": loss, "logits": logits, "past_key_values": presents}

    def _sample_next_token(self, logits, temperature, top_p, top_k):
        logits = logits / max(temperature, 1e-5)
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("inf")
        probs = F.softmax(logits, dim=-1)
        if top_p is not None:
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cumulative = torch.cumsum(sorted_probs, dim=-1)
            sorted_mask = cumulative - sorted_probs > top_p
            sorted_probs[sorted_mask] = 0.0
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_token = torch.multinomial(sorted_probs, num_samples=1)
            return torch.gather(sorted_idx, -1, next_token)
        return torch.multinomial(probs, num_samples=1)

    @torch.no_grad()
    def generate_batch(self, input_ids, attention_mask, max_new_tokens,
                        temperature=0.7, top_p=0.95, top_k=50, eos_id=None):
        self.eval()
        B, T0 = input_ids.shape
        assert T0 + max_new_tokens <= self.context_length, "prompt + new tokens exceed context_length"
        finished = torch.zeros(B, dtype=torch.bool, device=input_ids.device)

        out = self.forward(input_ids, attention_mask=attention_mask, use_cache=True, last_only=True)
        past, logits = out["past_key_values"], out["logits"][:, -1, :]

        for step in range(max_new_tokens):
            next_token = self._sample_next_token(logits, temperature, top_p, top_k)

            if eos_id is not None:
                next_token = torch.where(
                    finished.unsqueeze(1), torch.full_like(next_token, eos_id), next_token
                )

            input_ids = torch.cat([input_ids, next_token], dim=1)
            attention_mask = torch.cat([attention_mask, torch.ones_like(next_token)], dim=1)

            if eos_id is not None:
                finished = finished | (next_token.squeeze(1) == eos_id)
                if finished.all():
                    break
            if step == max_new_tokens - 1:
                break

            out = self.forward(next_token, attention_mask=attention_mask,
                               past_key_values=past, use_cache=True, last_only=True)
            past, logits = out["past_key_values"], out["logits"][:, -1, :]

        return input_ids


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

model = DecoderOnlyLM_RMSNorm_GQA(
    vocab_size=VOCAB_SIZE,          
    context_length=1536,
    n_embd=768, 
    n_layer=10,
    n_head=12,  
    n_kv_head=3,
    pad_id=PAD_ID,                 
    gradient_checkpointing=True,
)

n_params = count_parameters(model)
print(f"Model size: {n_params/1e6:.2f}M parameters")

In [ ]:
class BlockDataset(Dataset):
    def __init__(self, blocks):
        self.blocks = blocks

    def __len__(self):
        return len(self.blocks)

    def __getitem__(self, idx):
        return torch.tensor(self.blocks[idx], dtype=torch.long)

def collate_batch(batch):
    return torch.stack(batch, dim=0)

def build_optimizer(model, learning_rate, weight_decay):
    decay, no_decay = [], []
    for p in model.parameters():
        if not p.requires_grad:
            continue
        (no_decay if p.dim() < 2 else decay).append(p)

    return torch.optim.AdamW(
        [
            {"params": decay, "weight_decay": weight_decay},
            {"params": no_decay, "weight_decay": 0.0},
        ],
        lr=learning_rate,
        betas=(0.9, 0.95),
        eps=1e-8,
    )

def build_scheduler(optimizer, warmup_steps, total_steps):
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

@torch.no_grad()
def evaluate(model, val_loader, device, use_amp):
    model.eval()
    total_loss = 0.0
    total_batches = 0
    amp_dtype = torch.float16 if device.type == "cuda" else torch.bfloat16

    for input_ids in val_loader:
        input_ids = input_ids.to(device)
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
            out = model(input_ids, labels=input_ids)
        total_loss += out["loss"].item()
        total_batches += 1

    model.train()
    return total_loss / max(1, total_batches)


def save_checkpoint(path, model, optimizer, scheduler, scaler, epoch, batch_idx,global_step, log_history):
    torch.save(
        {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(),
            "epoch": epoch,
            "batch_idx": batch_idx,
            "global_step": global_step,
            "log_history": log_history,
        },
        path,
    )

def load_checkpoint(path, model, optimizer, scheduler, scaler, device):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    scaler.load_state_dict(ckpt["scaler"])
    epoch = ckpt["epoch"]
    batch_idx = ckpt["batch_idx"]
    global_step = ckpt["global_step"]
    log_history = ckpt.get("log_history", [])   

    print(f"\nRESUMING TRAINING\nEpoch       : {epoch + 1}\n"
          f"Batch       : {batch_idx}\nGlobal step : {global_step}\n")

    return epoch, batch_idx, global_step, log_history

def train_model(model, train_dataset, val_dataset, train_cfg, device, name,resume=True, save_every_seconds=600):
    last_ckpt_path = f"{name}_last_checkpoint.pt"
    final_model_path = f"{name}_final_model.pt"
    model.to(device)
    train_loader = DataLoader(
        train_dataset,
        batch_size=train_cfg.per_device_train_batch_size,
        shuffle=True,
        collate_fn=collate_batch,
        drop_last=True,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=train_cfg.per_device_eval_batch_size,
        shuffle=False,
        collate_fn=collate_batch,
        drop_last=False,
    )
    steps_per_epoch = len(train_loader) // train_cfg.gradient_accumulation_steps
    total_steps = max(1, steps_per_epoch * train_cfg.epochs)

    optimizer = build_optimizer(model, train_cfg.learning_rate, train_cfg.weight_decay)
    scheduler = build_scheduler(optimizer, train_cfg.warmup_steps, total_steps)

    use_amp = train_cfg.fp16 and device.type == "cuda"
    scaler = torch.amp.GradScaler(enabled=use_amp)
    amp_dtype = torch.float16 if device.type == "cuda" else torch.bfloat16

    start_epoch = 0
    start_batch = 0
    global_step = 0
    log_history = []

    if resume and os.path.exists(last_ckpt_path):
        start_epoch, start_batch, global_step, log_history = load_checkpoint(last_ckpt_path, model, optimizer, scheduler, scaler, device)

    model.train()

    running_loss = 0.0
    running_count = 0
    tokens_seen = 0

    start_time = time.time()
    last_save_time = time.time()

    for epoch in range(start_epoch, train_cfg.epochs):
        print(f"\nStarting Epoch {epoch + 1}/{train_cfg.epochs}")
        if epoch != start_epoch:
            start_batch = 0

        optimizer.zero_grad()
        pbar = tqdm(enumerate(train_loader), total=len(train_loader),
                    desc=f"Epoch {epoch + 1}/{train_cfg.epochs}")

        for batch_idx, input_ids in pbar:
            if batch_idx < start_batch:
                continue

            input_ids = input_ids.to(device)

            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
                out = model(input_ids, labels=input_ids)
                loss = out["loss"] / train_cfg.gradient_accumulation_steps

            scaler.scale(loss).backward()

            running_loss += loss.item() * train_cfg.gradient_accumulation_steps
            running_count += 1
            tokens_seen += input_ids.numel()

            if (batch_idx + 1) % train_cfg.gradient_accumulation_steps == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1

                if global_step % train_cfg.logging_steps == 0:
                    avg_loss = running_loss / max(1, running_count)
                    elapsed = time.time() - start_time
                    tps = tokens_seen / max(elapsed, 1e-6)
                    lr = scheduler.get_last_lr()[0]

                    pbar.set_postfix(loss=f"{avg_loss:.4f}", lr=f"{lr:.2e}", tok_s=f"{tps:,.0f}")

                    log_history.append({"step": global_step, "loss": avg_loss})

                    running_loss = 0.0
                    running_count = 0

                if global_step % train_cfg.eval_steps == 0:
                    eval_loss = evaluate(model, val_loader, device, use_amp)
                    pbar.write(f"[step {global_step}] eval_loss={eval_loss:.4f} "
                              f"perplexity={math.exp(min(eval_loss, 20)):.2f}")

                    log_history.append({"step": global_step, "eval_loss": eval_loss})

            if time.time() - last_save_time >= save_every_seconds:
                save_checkpoint(last_ckpt_path, model, optimizer, scheduler, scaler,
                                epoch, batch_idx + 1, global_step, log_history)
                last_save_time = time.time()
                pbar.write(f"Checkpoint saved (epoch {epoch + 1}, batch {batch_idx + 1})")

        print(f"Epoch {epoch + 1} completed.")
        save_checkpoint(last_ckpt_path, model, optimizer, scheduler, scaler, epoch + 1, 0, global_step, log_history)
        print(f"Epoch {epoch + 1} checkpoint saved.")

    save_checkpoint(last_ckpt_path, model, optimizer, scheduler, scaler,
                    train_cfg.epochs, 0, global_step, log_history)
    torch.save(model.state_dict(), final_model_path)

    with open(f"{name}_log_history.json", "w") as f:
        json.dump(log_history, f)

    print(f"\nTRAINING COMPLETE\nFinal model saved to: {final_model_path}")

    return log_history

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_ds = BlockDataset(train_blocks)  
val_ds = BlockDataset(val_blocks)

log_history = train_model(model, train_ds, val_ds, train_cfg, device,arch_cfg.name, resume=True)

In [ ]:
def load_test_prompts(hf_dataset_repo, test_file, hf_token):
    path = hf_hub_download(repo_id=hf_dataset_repo, filename=test_file,repo_type="dataset", token=hf_token)
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)   

def _encode_and_truncate(text, encode_fn, context_length, max_new_tokens):
    ids = encode_fn(text)
    max_prompt_len = context_length - max_new_tokens
    if len(ids) > max_prompt_len:
        ids = ids[-max_prompt_len:]
    return ids


def generate_continuations_batch(model, prompt_texts, encode_fn, decode_fn, bpe_data,device, context_length, target_words=128,max_new_tokens=384, temperature=0.7, top_p=0.95, top_k=50):
    eos_id = bpe_data["stoi"]["</s>"]
    pad_id = model.pad_id
    encoded = [_encode_and_truncate(t, encode_fn, context_length, max_new_tokens) for t in prompt_texts]
    max_len = max(len(ids) for ids in encoded)

    input_ids = torch.full((len(encoded), max_len), pad_id, dtype=torch.long, device=device)
    attention_mask = torch.zeros((len(encoded), max_len), dtype=torch.long, device=device)
    for i, ids in enumerate(encoded):
        input_ids[i, max_len - len(ids):] = torch.tensor(ids, dtype=torch.long, device=device)
        attention_mask[i, max_len - len(ids):] = 1

    generated_ids = model.generate_batch(input_ids, attention_mask, max_new_tokens=max_new_tokens,temperature=temperature, top_p=top_p, top_k=top_k, eos_id=eos_id)
    continuations = []
    for row in generated_ids[:, max_len:].tolist():
        if eos_id in row:
            row = row[:row.index(eos_id)]
        text = decode_fn(row, bpe_data)
        continuations.append(" ".join(text.split()[:target_words]))
    return continuations

def generate_all_predictions(model, test_data, encode_fn, decode_fn, bpe_data,device, context_length, gen_cfg, batch_size=16, progress=True):
    model.eval()
    predictions = {}
    items = list(test_data.items())
    batches = [items[i:i + batch_size] for i in range(0, len(items), batch_size)]
    pbar = tqdm(total=len(items), desc="Generating", unit="prompt", disable=not progress)
    for batch in batches:
        example_ids = [ex_id for ex_id, _ in batch]
        prompt_texts = [record["text"] for _, record in batch]
        continuations = generate_continuations_batch(
            model, prompt_texts, encode_fn, decode_fn, bpe_data, device, context_length,
            target_words=gen_cfg.target_generation_words,
            max_new_tokens=gen_cfg.max_generation_tokens,
            temperature=gen_cfg.temperature, top_p=gen_cfg.top_p, top_k=gen_cfg.top_k,
        )
        for example_id, continuation in zip(example_ids, continuations):
            predictions[str(example_id)] = {"text": continuation}
        pbar.update(len(batch))
    pbar.close()
    return predictions

PREDICTION_FILE = "prediction.json"
SUBMISSION_FILE = "submission_v7.zip"


def save_prediction_json(predictions):
    with open(PREDICTION_FILE, "w", encoding="utf-8") as f:
        json.dump(predictions, f, ensure_ascii=False, indent=2)


def create_submission_zip():
    with zipfile.ZipFile(SUBMISSION_FILE, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(PREDICTION_FILE, arcname="prediction.json")

def verify_submission_format(expected_ids):
    with zipfile.ZipFile(SUBMISSION_FILE, "r") as zf:
        names = zf.namelist()
        with zf.open("prediction.json") as f:
            preds = json.load(f)
    pred_ids = set(preds.keys())
    expected_ids = set(str(i) for i in expected_ids)
    missing = expected_ids - pred_ids
    extra = pred_ids - expected_ids

    print(f"Submission format OK: {len(preds)} predictions, all IDs correct, correct zip structure.")

In [ ]:
test_data = load_test_prompts(hf_dataset_repo, test_file, HF_TOKEN)
t = time.time()
with torch.no_grad():
    predictions = generate_all_predictions(model, test_data, make_encoder(bpe_data), decode, bpe_data,device, arch_cfg.context_length, gen_cfg, batch_size=16)
    
print(f"{time.time() - t:.1f}s for {len(predictions)} prompts")

save_prediction_json(predictions)
create_submission_zip()
verify_submission_format(expected_ids=test_data.keys())